# How the Fluid-Film Journal Bearing Solver Works

This page explains what `FluidFilmBearing` and its configuration classes (`PlainJournal`,
`TiltingPad`, `EllipticalBearing`, ...) compute between the bearing description and the
coefficient table that the rotor uses: what one case solves, how its convergence is judged, how
the pad degrees of freedom are condensed at the whirl frequency, which viscosity laws the
lubricant can follow and how cavitation is modelled. How to use the classes is shown in the
[fluid-film bearings tutorial](../user_guide/tutorial_bearings_part_2).

The examples use a five-pad tilting-pad bearing with the load between pads and the isoviscous
film model, which solves in about a second.

## 1. One case per speed

The element solves one case per entry of `speed`:

```
FluidFilmBearing.__init__        validates the inputs, resolves the lubricant
 └─ BearingSetup.from_inputs     ross/bearings/fluid_film/sweep.py: meshes and pad geometry, once
     └─ solve_sweep(setup, speeds, warm_start, chunk_size, n_jobs)
         └─ one case per speed   equilibrium -> stiffness and damping blocks -> PadMatrices
 └─ FluidFilmBearingResults      fields, outputs, convergence records, operating_results
 └─ BearingElement               coefficient table condensed at the whirl frequency of each entry
```

A case is a set of nested fixed-point loops around a Newton search for the journal position that
balances the applied load (`equilibrium_type="match_load"`): inside each Newton iteration the pad
tilts are found from the pivot moment balance, and around it the turbulence, film and pad
temperature, inlet mixing, deformation, pivot and sump loops run when the model uses them. At the
converged state the engine perturbs the film to get the stiffness and damping blocks of every pad
(Section 4).

The speeds are sorted and cut into chains of `chunk_size` consecutive speeds (default 5). The first
speed of a chain starts from `initial_position`; each later speed starts from the converged state
of the previous one (journal position, pad tilts, temperature and viscosity fields, deformations)
and its Newton search takes full steps (`warm_start=True`, the default). A warm-started case that
fails is solved again cold. A warm-started speed converges to the same tolerances as a cold one but
from another starting point, so its coefficients differ from an independent solve within those
tolerances, and the coefficient at one speed depends slightly on the other speeds of the sweep;
`warm_start=False` solves every speed independently, and each then equals a bearing built for that
speed alone. `n_jobs` sets how many chains run at once in worker processes; the chains depend only
on the speeds and every worker runs with single-threaded BLAS, so the results do not depend on
`n_jobs`. The workers are started with `spawn`: a script that uses `n_jobs > 1` must call the solve
under an `if __name__ == "__main__":` guard (a notebook needs none). If the workers stop before they
return their results, a `RuntimeError` is raised; there is no fallback to a serial solve.

With `frequency=` the element holds a 2-D (speed, frequency) table, but each speed is still solved
once: the state of the speed is condensed at every frequency (Section 4).

In [1]:
import warnings

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

import ross as rs
from ross.units import Q_

pio.renderers.default = "notebook_connected"

tilting_pad = dict(
    n=1,
    equilibrium_type="match_load",
    fxs_load=884.05,
    fys_load=-2670.4,
    journal_diameter=101.6e-3,
    radial_clearance=74.9e-6,
    pad_thickness=12.7e-3,
    pivot_angle=Q_([18, 90, 162, 234, 306], "deg"),
    pad_arc=Q_([60] * 5, "deg"),
    pad_axial_length=[50.8e-3] * 5,
    preload=[0.5] * 5,
    offset=[0.5] * 5,
    lubricant="ISOVG32",
    oil_supply_temperature=Q_(40, "degC"),
    oil_flow_v=Q_(10, "l/min"),
    thermal_type=None,
)
bearing = rs.TiltingPad(speed=Q_([3000, 4000, 5000], "RPM"), **tilting_pad)
results = bearing.results

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Convergence diagnostics

Every loop of a case reports its last pass: iterations, residual, tolerance and, when it did not
converge, a reason. A case is converged when every loop that ran ended below its tolerance, the
case stayed within its work budget and the outputs are finite.

| Loop | Residual | Tolerance |
|---|---|---|
| `newton` | force balance $\lvert F_\mathrm{net}\rvert / \lvert F_\mathrm{ext}\rvert$ | $2\times10^{-3}$ |
| `tilt` | worst pad: the smaller of $\lvert M\rvert / (10^{-9} M_\mathrm{scale})$ and the root bracket over $10^{-10}$ rad | 1 |
| `turbulence` | RMS change of the wall shear stress, Pa | engine constant |
| `viscosity`, `mixing`, `journal_temperature`, `sump` | temperature changes, K | engine constants |
| `elasto`, `pivot` | deformation changes over the clearance $C_b$ | engine constant, 0.01 |
| `film_onset` | pads whose film onset moved (starved film); with the mass-conserving film, the change of the inlet film fraction | 0 / $10^{-6}$ |
| `cavitation` | scaled residual of the mass-conserving film solve (Section 6) | $10^{-13}$ |

A loop the model does not use records one pass with a zero residual. Each case also runs under a
budget of Reynolds solves, `max_reynolds_solves` (default 400,000 per pad, counting at least 5
pads), with a separate cap of 4,000 Reynolds solves per pad on one equilibrium solve, and an optional
wall-clock `time_limit`; a case that reaches it stops with reason
`"budget"` or `"time_limit"`, and its outputs are evaluated at the last journal position. Other
reasons are `"max_iterations"`, `"diverged"`, `"singular_jacobian"`, `"pivot_unloaded"`,
`"non_finite"` and `"contact"` (the journal position crosses a pad surface), and, for the
mass-conserving film solve, `"stalled"`, `"singular"` and `"nan"`. A thermal case (`"adiabatic"` or
`"full"`) whose equilibrium pass puts the journal through a pad stops there with `"contact"`: the
energy equation is not solved on a closed film, and the outlet temperatures are NaN.

The records are in `results.convergence`, one dictionary per case:

In [2]:
record = results.convergence[1]
print("converged:", record["converged"], "| started:", record["warm_start"]["start"])
print("Reynolds solves:", record["reynolds_solves"])
for name in ("newton", "tilt"):
    loop = record["loops"][name]
    print(
        f"{name:>6}: {loop['iterations']} iterations, residual {loop['residual']:.3g}, "
        f"tolerance {loop['tolerance']:.3g}, {loop['reason']}"
    )

converged: True | started: warm
Reynolds solves: 67
newton: 3 iterations, residual 0.000148, tolerance 0.002, converged
  tilt: 6 iterations, residual 0.5, tolerance 1, converged


A case that does not converge raises a `ConvergenceWarning`, and `on_failure` decides what happens
to its speed: `"warn"` (default) leaves it out of the coefficient table, so the element interpolates
between the converged speeds; `"raise"` raises `FluidFilmConvergenceError`; `"keep"` keeps the row.
Here a budget of 40 Reynolds solves stops the case before its equilibrium is found:

In [3]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    small_budget = rs.TiltingPad(
        speed=Q_([3000], "RPM"), max_reynolds_solves=40, on_failure="keep", **tilting_pad
    )
print(caught[0].category.__name__, ":", str(caught[0].message)[:120], "...")
record = small_budget.results.convergence[0]
print("reason:", record["reason"], "| Reynolds solves:", record["reynolds_solves"])

reason: budget | Reynolds solves: 40


## 3. Operating results

`bearing.operating_results` is a `pandas.DataFrame` with one row per case, converged or not, with
the quantities an analyst checks before using the coefficients:

- position: `eccentricity` (over the bearing clearance $C_b$), `attitude` (from $-y$ towards $+x$)
  and `attitude_from_load`, `xj`, `yj`;
- film: minimum film thickness `h_min` and its pad, peak pressure `p_max`, peak pad and film
  temperatures;
- power and flows: `power_loss`, `q_supply`, `q_side`, `q_carryover`;
- dimensionless groups: the Sommerfeld number
  $S = \dfrac{\mu N L D}{W} \left(\dfrac{R}{C_b}\right)^2$ ($N$ in rev/s, $\mu$ at the supply
  temperature), the specific load $W/(LD)$, the largest local film Reynolds number
  $\mathrm{Re}_\mathrm{max} = \max \rho U h / \mu$, the clearance Reynolds number
  $\mathrm{Re}_{C_b} = \rho U C_b / \mu$ and Taylor's onset of vortex flow
  $\mathrm{Re}_\mathrm{Taylor} = 41.2\sqrt{R/C_b}$ {cite}`taylor1923`;
- per pad: the film load `pad_load` and the `tilt_angle`.

`re_max` is the number the turbulence model compares with `re_laminar` / `re_turbulent`; on a
preloaded pad the local film reaches about $2C_b$, so `re_max` exceeds `re_cb`. The flow is laminar
below `re_laminar` (default 500), turbulent above `re_turbulent` and blended between them.
`re_turbulent=None` (default) resolves to 1000 with a numerical `re_laminar`. With
`re_laminar="taylor"` the laminar limit is Taylor's onset $\mathrm{Re}_\mathrm{Taylor}$ and
`re_turbulent=None` resolves to $2\,\mathrm{Re}_\mathrm{Taylor}$. Two choices of this option are
not part of Taylor's criterion: the factor 2 of the turbulent limit, and the comparison of both
limits with the local `re_max` rather than with a Reynolds number on the mean clearance, which
brings the onset earlier on preloaded pads. An explicit `re_turbulent` is always used as given.

In [4]:
columns = ["speed", "converged", "eccentricity", "attitude", "h_min", "p_max",
           "power_loss", "sommerfeld", "re_max", "re_cb", "re_taylor"]
bearing.operating_results[columns]

,speed,converged,eccentricity,attitude,h_min,p_max,power_loss,sommerfeld,re_max,re_cb,re_taylor
0,314.159265,True,0.338828,0.304484,0.000043,2.399385e+06,1372.460301,1.180299,61.979824,37.354698,1072.971443
1,418.879020,True,0.273894,0.312202,0.000047,2.558124e+06,2385.228419,1.573731,80.261079,49.806265,1072.971443
2,523.598776,True,0.228402,0.315337,0.000050,2.754054e+06,3680.586812,1.967164,98.224882,62.257831,1072.971443


## 4. Frequency dependence: pad matrices and exact condensation

A tilting pad adds its own coordinates to the journal coordinates $u = (x, y)$: the tilt
$\delta_k$ about the pivot and, with a flexible pivot, the pivot displacement $v_k$. For each
solved speed the engine keeps, for every pad $k$, the unreduced stiffness, damping and mass
matrices on $[u;\ \eta_k]$, with $\eta_k$ the pad coordinates. The pad coordinates are not rotor
coordinates, so the rotor receives them condensed at the whirl frequency $\omega$
{cite}`lund1964,dimond2011,wilkes2012`:

$$
Z_k(s) = K_k + s C_k + s^2 M_k, \qquad
D(s) = \sum_k \left[ Z_{uu} - Z_{u\eta} Z_{\eta\eta}^{-1} Z_{\eta u} \right]_k ,
$$

$$
K(\omega) = \operatorname{Re} D(i\omega), \qquad C(\omega) = \operatorname{Im} D(i\omega) / \omega .
$$

At $\omega = 0$ the static condensation applies,
$K(0) = K_{uu} - K_{u\eta} K_{\eta\eta}^{-1} K_{\eta u}$ (and the matching first-order term for
$C(0)$); as $\omega \to \infty$ with pad inertia, $K_\infty = K_{uu} - C_{u\eta} M_{\eta\eta}^{-1}
C_{\eta u}$ and $C_\infty = C_{uu}$. The frozen values $\sum_k K_{uu}$, $\sum_k C_{uu}$ (pads held
fixed) are not a limit of $D$. A fixed-geometry bearing has no pad coordinates, so its coefficients
do not depend on $\omega$.

**Pad inertia.** Each pad is an annular sector of bore radius $R_i = R + C_p$, back radius
$R_o = R_i + t$, arc $\beta$ and length $L$, with its pivot at radius $r_P = R + t$ and angle
$\phi = (\mathrm{offset} - 1/2)\beta$ from the pad centre line. With $r_g$ the radius of the
centroid,

$$
m = \rho L \beta \frac{R_o^2 - R_i^2}{2}, \quad
I_O = m \frac{R_o^2 + R_i^2}{2}, \quad
r_g = \frac{4 \sin(\beta/2) (R_o^3 - R_i^3)}{3\beta (R_o^2 - R_i^2)},
$$

$$
I_P = I_O + m r_P^2 - 2 m r_P r_g \cos\phi, \qquad c = m r_g \sin\phi ,
$$

so the kinetic energy of a pad that tilts by $\delta$ about the pivot and moves by $v$ along the
pivot direction is $\tfrac12 [\dot\delta, \dot v]\,[[I_P, c], [c, m]]\,[\dot\delta, \dot v]^T$; a
centred pivot has no coupling. With a flexible pivot the film sees the journal relative to the
pivot, $u - v\,n_k$, so $K_k = T^T K_\mathrm{film} T + \mathrm{diag}(0, 0, k_\mathrm{rot}, k_\mathrm{pivot})$
with $T$ the map from $[u, \delta, v]$ to the film coordinates.

`results.pad_matrices(i)` returns these matrices for speed `i` (`PadMatrices`,
`ross/bearings/fluid_film/padmatrices.py`):

In [5]:
pads = results.pad_matrices(0)
print("coordinates of pad 0:", pads.labels[0])
print(f"pad mass {pads.pad_mass[0]:.4f} kg, inertia about the pivot {pads.pad_inertia[0]:.4e} kg m^2")

coordinates of pad 0: ('x', 'y', 'tilt_0')
pad mass 0.3031 kg, inertia about the pivot 1.1467e-04 kg m^2


`results.coefficients(i, frequency)` condenses them at any whirl frequency without a new solve
(the element's own `coefficients()` interpolates the table, as the rotor does);
`frequency=0` gives the static condensation. Below, the condensed direct coefficients of
the first speed over the whirl ratio, with the frozen values for comparison. In this calculation
the direct stiffness falls and the direct damping rises with the whirl frequency over the whole
range (checked in the cell).

In [6]:
speed = results.frequency[0]
ratios = np.linspace(0.0, 3.0, 61)
K, C = results.coefficients(0, frequency=ratios * speed)
K_frozen = sum(k[:2, :2] for k in pads.K)
C_frozen = sum(c[:2, :2] for c in pads.C)

fig = go.Figure()
for name, values, frozen, unit in (
    ("kyy", K[:, 1, 1], K_frozen[1, 1], 1e6),
    ("kxx", K[:, 0, 0], K_frozen[0, 0], 1e6),
):
    fig.add_scatter(x=ratios, y=values / unit, name=f"{name} condensed")
    fig.add_scatter(x=ratios, y=np.full_like(ratios, frozen / unit), name=f"{name} frozen",
                    line=dict(dash="dash"))
for name, values, frozen in (("cyy", C[:, 1, 1], C_frozen[1, 1]), ("cxx", C[:, 0, 0], C_frozen[0, 0])):
    fig.add_scatter(x=ratios, y=values / 1e3, name=f"{name} condensed", yaxis="y2")
fig.update_layout(
    xaxis_title="whirl frequency / speed",
    yaxis_title="stiffness [MN/m]",
    yaxis2=dict(title="damping [kN s/m]", overlaying="y", side="right"),
    title="Condensed coefficients of the five-pad bearing at 3000 rpm",
)
for name, values, trend in (
    ("kxx", K[:, 0, 0], -1), ("kyy", K[:, 1, 1], -1), ("cxx", C[:, 0, 0], 1), ("cyy", C[:, 1, 1], 1)
):
    word = "falls" if trend < 0 else "rises"
    print(f"{name} {word} monotonically over 0-3X: {bool(np.all(trend * np.diff(values) > 0))}")
fig

kxx falls monotonically over 0-3X: True
kyy falls monotonically over 0-3X: True
cxx rises monotonically over 0-3X: True
cyy rises monotonically over 0-3X: True


For a damped rotor eigenvalue $s = \lambda$ the exact bearing force is $D(\lambda)$, which
`pads.impedance(s)` returns. `run_modal(matched_whirl=True)` uses $K(\omega) + i\omega C(\omega)$
at the damped natural frequency $\omega = \operatorname{Im}\lambda$, which equals $D(\lambda)$ only
for an undamped mode:

In [7]:
omega = speed
for zeta in (0.0, 0.1, 0.3):
    s = omega * (-zeta + 1j * np.sqrt(1 - zeta**2))
    exact = pads.impedance(s)
    Kw, Cw = results.coefficients(0, frequency=s.imag)
    matched = Kw + 1j * s.imag * Cw
    gap = np.abs(matched - exact).max() / np.abs(exact).max()
    print(f"damping ratio {zeta}: |K + i w C - D(s)| / |D(s)| = {gap:.2e}")

damping ratio 0.0: |K + i w C - D(s)| / |D(s)| = 0.00e+00
damping ratio 0.1: |K + i w C - D(s)| / |D(s)| = 1.04e-01
damping ratio 0.3: |K + i w C - D(s)| / |D(s)| = 3.97e-01


The static limit is `results.coefficients(i, frequency=0)`, which equals
`pads.condense_static()`; its damping is finite. `pads.condense_limit()` gives the closed form of
the high-frequency limit. It is the asymptote of the condensation, **not a stiffness to use in a
rotor model**: it is approached only far above any whirl frequency a rotor reaches, and its stiffness
can have the opposite sign of the static value and a much larger magnitude, as here:

In [8]:
K0, C0 = results.coefficients(0, frequency=0.0)
Ks, Cs = pads.condense_static()
print("frequency=0 equals condense_static:", np.array_equal(K0, Ks) and np.array_equal(C0, Cs))
K_inf, C_inf = pads.condense_limit()
for name, i in (("kxx", 0), ("kyy", 1)):
    print(f"{name}: static {K0[i, i]:.4e} N/m, high-frequency asymptote {K_inf[i, i]:.4e} N/m "
          f"(ratio {K_inf[i, i] / K0[i, i]:.0f})")

frequency=0 equals condense_static: True
kxx: static 1.1912e+08 N/m, high-frequency asymptote -6.5910e+10 N/m (ratio -553)
kyy: static 1.5038e+08 N/m, high-frequency asymptote -5.2203e+10 N/m (ratio -347)


## 5. Lubricant laws

The journal bearings resolve the lubricant into a `ross.bearings.film.lubricant.Lubricant`. Two
viscosity laws pass through two table points $(T_1, \mu_1)$, $(T_2, \mu_2)$:

- exponential (`viscosity_model="exponential"`, the default for a lubricant name or dictionary,
  and the law of every result before ROSS 3.2): $\mu = \mu_1 \exp[\beta (T - T_1)]$,
  $\beta = \ln(\mu_2/\mu_1)/(T_2 - T_1)$;
- Walther, ASTM D341 {cite}`astm_d341` (`viscosity_model="walther"`):
  $\log_{10}\log_{10}(\nu + 0.7) = A - B \log_{10} T$, with the kinematic viscosity
  $\nu = \mu/\rho(T)$ in mm²/s and $T$ in K.

A `Lubricant` instance passed as `lubricant` keeps its own law. The two laws agree at the table
points and differ between and outside them (table below): between the points the Walther law gives
a lower viscosity than the exponential fit, so a thermal model with Walther runs a cooler film with
less power loss. The density, specific heat and conductivity are linear in the temperature,
$\rho(T) = \rho_1 [1 - a_v (T - T_1)]$, $c_p(T) = c_1 + s_c (T - T_1)$, constant by default; the
journal energy equation takes them as constants, evaluated at the supply temperature.

`Lubricant.from_datasheet(nu40, nu100, rho15)` builds a lubricant from an oil datasheet: the
kinematic viscosities at 40 and 100 °C and the density at 15 °C, with the Walther law by default.

- The density follows the form given for lubricating oils in API MPMS 11.1 {cite}`api_mpms_11_1`,
  $\rho(T) = \rho_{15} \exp[-a_{15}\Delta T (1 + 0.8 a_{15}\Delta T)]$, $a_{15} = K_1/\rho_{15}$,
  $K_1 = 0.6278$ kg/(m³ K), $\Delta T = T - 288.15$ K, linearised at 40 °C. The form and the
  constant $K_1$ have not been checked against the standard.
- The specific heat and the conductivity follow Cragoe {cite}`cragoe1929` (equations 5 and 4 of
  the publication, converted to SI and rounded): $c_p = (1684 + 3.39\,t)/\sqrt{d}$ J/(kg K) and
  $k = 0.1172 (1 - 0.00054\,t)/d$ W/(m K), with $t$ in °C and the specific gravity
  $d = \rho_{15}/999$. The publication estimates their accuracy at 5 % and 10 %.

The cell compares the two laws for ISO VG32 with a datasheet oil, and the ISO VG32 table entry with
the Cragoe correlations:

In [9]:
from ross.bearings.film.lubricant import (
    Lubricant,
    cragoe_specific_heat,
    cragoe_thermal_conductivity,
)

exponential = Lubricant.from_name("ISOVG32", viscosity_model="exponential")
walther = Lubricant.from_name("ISOVG32", viscosity_model="walther")
datasheet = Lubricant.from_datasheet(Q_(32, "cSt"), Q_(5.4, "cSt"), 870.0)

print(" T [C]  exponential  Walther  exp/Walther-1  datasheet  [mPa s]")
for t in (20, 30, 40, 50, 60, 70, 80, 100, 120):
    T = t + 273.15
    mu_e, mu_w = exponential.viscosity(T), walther.viscosity(T)
    print(f"{t:6d} {mu_e * 1e3:12.2f} {mu_w * 1e3:8.2f} {100 * (mu_e / mu_w - 1):+13.1f} %"
          f" {datasheet.viscosity(T) * 1e3:10.2f}")
print(f"datasheet oil: rho(40 C) = {datasheet.density:.1f} kg/m^3, "
      f"c_p = {datasheet.specific_heat:.0f} J/(kg K), k = {datasheet.thermal_conductivity:.4f} W/(m K)")

d, T40 = 0.8718, 313.15
print(f"ISOVG32 table: c_p = {exponential.specific_heat:.1f} J/(kg K), "
      f"k = {exponential.thermal_conductivity:.5f} W/(m K)")
print(f"Cragoe at d = {d}, 40 C: c_p = {cragoe_specific_heat(d, T40):.1f} J/(kg K), "
      f"k = {cragoe_thermal_conductivity(d, T40):.5f} W/(m K)")

 T [C]  exponential  Walther  exp/Walther-1  datasheet  [mPa s]
    20        50.80    76.11         -33.3 %      74.79
    30        37.69    44.43         -15.2 %      43.57
    40        27.97    27.97          +0.0 %      27.34
    50        20.75    18.73         +10.8 %      18.23
    60        15.40    13.20         +16.7 %      12.78
    70        11.42     9.70         +17.7 %       9.34
    80         8.48     7.39         +14.7 %       7.07
   100         4.67     4.67          -0.0 %       4.41
   120         2.57     3.21         -20.1 %       2.99
datasheet oil: rho(40 C) = 854.2 kg/m^3, c_p = 1950 J/(kg K), k = 0.1317 W/(m K)
ISOVG32 table: c_p = 1948.8 J/(kg K), k = 0.13126 W/(m K)
Cragoe at d = 0.8718, 40 C: c_p = 1948.8 J/(kg K), k = 0.13153 W/(m K)


The specific heat of the ISO VG32 entry is the value of the Cragoe correlation at a specific
gravity of 0.8718, and its conductivity differs from it by 0.2 %, so the table and the correlations
are consistent rather than independent.

## 6. Cavitation models

The divergent part of a film cannot sustain a pressure much below the ambient: the film ruptures
into streamers of liquid separated by gas and vapour. `cavitation_model` selects how the film
equation treats it.

**Reynolds model** (`"reynolds"`, default). The pressure of the finite-element Reynolds solution
is bounded below by `cavitation_pressure` (the Reynolds, or Swift-Stieber, condition), and the
divergent clearance downstream of the minimum film is held at it. Mass is not conserved across the
cavitated zone: the film never reforms within a pad.

**Mass-conserving model** (`"mass_conserving"`). Each pad's film is the Elrod-Adams film
{cite}`elrod1981` written as a complementarity problem {cite}`giacopini2010`. The unknowns are the
pressure $p$ and the film fraction $\theta$ (the fraction of the clearance filled with liquid);
the mass flux per unit width is

$$
\mathbf{q} = -A \nabla p + c\, h\, \theta\, \mathbf{e}_x, \qquad \nabla \cdot \mathbf{q} = 0,
$$

$$
p \ge p_\mathrm{cav}, \qquad \theta \le 1, \qquad (p - p_\mathrm{cav})(1 - \theta) = 0,
$$

with $A = -h^3\Gamma$ and $c = U G$ from the engine's cross-film viscosity integrals ($h^3/(12\mu)$
and $U/2$ for a laminar isoviscous film). The film is full where the pressure is above
$p_\mathrm{cav}$; in the cavitated zone the pressure is $p_\mathrm{cav}$ and the oil crosses the pad
as a striated film carried by the Couette flow, so rupture and reformation (the
Jakobsson-Floberg-Olsson conditions {cite}`jakobsson1957,olsson1965`) follow from the flux balance.

The equations are discretised with finite volumes centred on the Reynolds nodes: central
diffusion and the Couette flux $(c\,h)_f\,\theta_\mathrm{up}$ with the film fraction of the upstream
node {cite}`ausas2007`, second order in the full film and first order in the film fraction of a
cavitated zone. The solver follows the approach of Woloszynski et al. {cite}`woloszynski2015`: the
complementarity is written with the Fischer-Burmeister function and solved by a Newton method with a
Schur elimination of the film fraction, one sparse factorisation per iteration. When the merit stops
improving, the iteration returns to its best iterate (a watchdog {cite}`chamberlain1982`) and
continues with primal-dual active-set steps {cite}`hintermuller2002`. Other algorithms of this family are those of Elrod {cite}`elrod1981` and
Vijayaraghavan and Keith {cite}`vijayaraghavan1989`.

In the journal engine:

- the oil enters a pad full (flooded) or, with `operating_type="starved_condition_even"`, with the
  inlet film fraction at which the pad's net draw through its leading edge equals the available oil
  (supply share plus hot-oil carry-over); a pad that draws less floods and spills the rest. The
  film onset stays at the leading edge;
- the pad flows (inlet, trailing edge, side leakage) come from the fluxes of the finite volumes, so
  a starved bearing drains exactly its supply, and the hot-oil carry-over is the trailing-edge
  outflow;
- the stiffness blocks are central differences of the mass-conserving film force (tilts held for
  the journal block, journal held for the tilt blocks); the damping blocks are the Reynolds
  perturbation with the cavitated nodes held, which omits the change of the film content with the
  squeeze velocity, so the damping is approximate;
- it supports `thermal_type` None or `"adiabatic"` (the shear dissipation is scaled by the film
  fraction), rigid pads (`deform_type=None`), flooded operation, and evenly starved operation on
  fixed-geometry bearings only. Other combinations (the full thermal model, pad deformation,
  starved tilting pads, other operating types, the pressure dam, a 360 deg pad) raise
  `NotImplementedError` before any solve;
- a film that reforms inside a land converges at first order in the circumferential element size,
  so such films need a fine circumferential mesh.

The cells below use the two-axial-groove journal of `fluid_film_bearing_example()`. The function
takes no arguments; the element keeps its constructor inputs, so the cell reads them back, leaves
out the example's coarse mesh (the class defaults then apply, 40 x 20 film elements and 30 across
the film) and overrides only the mesh and, later, the supply. The first cell solves the flooded journal with both
models on four meshes. The eccentricity gap falls with the mesh, but the coefficient gaps do not fall
steadily. Part of that is the default model's own mesh dependence: between successive meshes its
coefficients move by amounts of the same order as the gaps (second table). The cause of the rest is
not known:

In [10]:
import inspect

from ross.bearings.fluid_film_bearing import fluid_film_bearing_example

example = fluid_film_bearing_example()
parameters = inspect.signature(rs.FluidFilmBearing).parameters
journal = {
    name: getattr(example, name)
    for name in parameters
    if name not in ("kwargs", "num_processes") and not name.startswith("total_")
}
keys = ["kxx", "kxy", "kyx", "kyy", "cxx", "cxy", "cyy"]


def coefficient(bearing, key):
    return np.ravel(getattr(bearing, key))[0]


def compare(**kwargs):
    default = rs.FluidFilmBearing(**(journal | kwargs))
    mass_conserving = rs.FluidFilmBearing(**(journal | kwargs | {"cavitation_model": "mass_conserving"}))
    ecc = [b.operating_results["eccentricity"].iloc[0] for b in (default, mass_conserving)]
    gaps = []
    for k in keys:
        new, ref = coefficient(mass_conserving, k), coefficient(default, k)
        gaps.append(100 * (new - ref) / abs(ref))
    return default, mass_conserving, 100 * (ecc[1] / ecc[0] - 1), gaps


meshes = ((20, 10), (40, 20), (80, 40), (160, 80))
solved = []
print("(mass-conserving - default) / |default|, %")
print(f"{'mesh':>8} {'ecc':>7}" + "".join(f"{k:>7}" for k in keys))
for nx, nz in meshes:
    default, mass_conserving, ecc_gap, gaps = compare(total_ex_film=nx, total_ez_film=nz)
    solved.append((default, mass_conserving))
    print(f"{nx:>4}x{nz:<3} {ecc_gap:+7.3f}" + "".join(f"{g:+7.2f}" for g in gaps))

print("\nchange from the previous mesh, (new - old) / |old|, %")
print(f"{'mesh':>8} {'model':>16}" + "".join(f"{k:>7}" for k in keys))
for (nx, nz), (old, new) in zip(meshes[1:], zip(solved, solved[1:])):
    for label, model in (("default", 0), ("mass-conserving", 1)):
        changes = [
            100 * (coefficient(new[model], k) - coefficient(old[model], k))
            / abs(coefficient(old[model], k))
            for k in keys
        ]
        print(f"{nx:>4}x{nz:<3} {label:>16}" + "".join(f"{c:+7.2f}" for c in changes))

(mass-conserving - default) / |default|, %
    mesh     ecc    kxx    kxy    kyx    kyy    cxx    cxy    cyy
  20x10   +0.087  +2.11  -3.21  -1.12  +1.64  +0.35  -0.39  +0.53


  40x20   -0.025  +0.31  -1.39  -0.09  +0.41  -0.03  +0.07  +0.03


  80x40   +0.004  +0.80  +2.29  -0.33  -0.38  +1.73  -1.74  +0.45


 160x80   -0.000  +0.25  +0.76  -0.10  -0.13  +0.58  -0.58  +0.15

change from the previous mesh, (new - old) / |old|, %
    mesh            model    kxx    kxy    kyx    kyy    cxx    cxy    cyy
  40x20           default  +1.23  -3.72  -0.79  +1.54  -2.55  +2.89  -0.54
  40x20   mass-conserving  -0.55  -1.91  +0.24  +0.31  -2.92  +3.34  -1.04
  80x40           default  -0.35  -3.57  +0.13  +0.90  -2.33  +2.41  -0.54
  80x40   mass-conserving  +0.13  +0.04  -0.11  +0.11  -0.61  +0.65  -0.13
 160x80           default  +0.36  +0.77  -0.15  -0.11  +0.39  -0.38  +0.11
 160x80   mass-conserving  -0.19  -0.74  +0.08  +0.14  -0.74  +0.75  -0.19


Under starved supply the two models differ in kind. The default model moves the film onset into the
pad until the pad draws the available oil; the mass-conserving film keeps the onset at the leading
edge and lowers the inlet film fraction instead. The cell solves the same journal at 12 and 8 L/min
on the default 40 x 20 mesh, with carry-over 0.8, and prints the cross-coupled stiffness of the default model
relative to its direct terms, against which the large relative changes of `kxy` should be read:

In [11]:
starved_inputs = dict(
    operating_type="starved_condition_even",
    hot_oil_lambda=0.8,
)
print(f"{'supply':>8} {'ecc':>7}" + "".join(f"{k:>8}" for k in keys)
      + "  side leakage default / mass-conserving")
starved_cases = []
for supply in (12, 8):
    default, starved, ecc_gap, gaps = compare(oil_flow_v=Q_(supply, "l/min"), **starved_inputs)
    starved_cases.append((supply, default, starved))
    q = [b.operating_results["q_side"].iloc[0] * 6e4 for b in (default, starved)]
    print(f"{supply:>4} L/min {ecc_gap:+6.1f}%" + "".join(f"{g:+7.1f}%" for g in gaps)
          + f"  {q[0]:.2f} / {q[1]:.2f} L/min")
for supply, default, starved in starved_cases:
    kxx, kxy, kyy = (coefficient(default, k) for k in ("kxx", "kxy", "kyy"))
    change = coefficient(starved, "kxy") - kxy
    print(f"default model at {supply} L/min: kxy / kxx = {kxy / kxx:+.3f}, kxy / kyy = {kxy / kyy:+.3f};"
          f" change of kxy / kxx = {change / kxx:+.2f}")

  supply     ecc     kxx     kxy     kyx     kyy     cxx     cxy     cyy  side leakage default / mass-conserving


  12 L/min   -2.4%  -14.2% +237.7%   -9.2%  +33.0%  +12.0%   -7.8%   -4.7%  6.23 / 6.30 L/min


   8 L/min   -0.8%  -31.9% +520.6%  -31.9%  +76.3%  +14.3%  -22.0%   +0.3%  3.82 / 3.30 L/min
default model at 12 L/min: kxy / kxx = +0.302, kxy / kyy = +0.071; change of kxy / kxx = +0.72
default model at 8 L/min: kxy / kxx = +0.265, kxy / kyy = +0.040; change of kxy / kxx = +1.38


The eccentricity changes by 2.4 % at 12 L/min and 0.8 % at 8 L/min, but the coefficients change
much more. The direct stiffness changes by -14.2 % (`kxx`) and +33.0 % (`kyy`) at 12 L/min, and by
-31.9 % and +76.3 % at 8 L/min. The cross-coupled stiffness `kxy` rises by 238 % and 521 %, and
`kyx` falls by 9.2 % and 31.9 %. In the default model `kxy` is 0.30 and 0.27 of `kxx` (0.07 and
0.04 of `kyy`); in absolute terms the change of `kxy` is 0.72 and 1.38 times `kxx`.

On the default mesh the flooded journal reports, with the mass-conserving film, the iterations and
the mass balance of the last film solve of each pad:

In [12]:
mass_conserving = rs.FluidFilmBearing(
    **(journal | dict(cavitation_model="mass_conserving"))
)
film = mass_conserving.results.convergence[0]["loops"]["cavitation"]
print("film solver:", film["iterations"], "iterations, mass balance per pad", np.array(film["mass_balance"]))

film solver: 6 iterations, mass balance per pad [1.87350281e-16 1.00953034e-15]


`plot_film_fraction_2d` draws the film fraction along the axial centre plane. On the unloaded
upper pad the film is cavitated from the leading edge, and the film fraction follows the clearance
so that the striated film carries the oil that entered: $h\,\theta$ is constant along the pad.

In [13]:
mass_conserving.plot_film_fraction_2d()

## References

```{bibliography}
:filter: docname in docnames
```